# Artificial Neural Network — GAT IOP Prediction

This notebook develops an Artificial Neural Network (ANN) to predict Applanation/GAT IOP from:

- iCare IOP
- Central Corneal Thickness (CCT)

The same patient-level train, validation, and test datasets used for the Random Forest are used here to allow a fair comparison between models.

The test dataset will remain untouched during model development and hyperparameter tuning.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [3]:
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.13.0+cu130
GPU available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
## Load Dataset

The ANN uses the same training, validation, and test splits as the Random Forest.

Features:
- iCare IOP
- CCT

Target:
- Applanation/GAT IOP

In [4]:
train = pd.read_csv("../data/splits/train.csv")
validation = pd.read_csv("../data/splits/validation.csv")
test = pd.read_csv("../data/splits/test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (4648, 5)
Validation: (1001, 5)
Test: (990, 5)


## Define Features and Target

The ANN receives two input features:

X = [iCare IOP, CCT]

The target is:

y = Applanation/GAT IOP

In [16]:
features = ["icare_iop", "cct"]
target = "applanation_iop"

X_train = train[features].values
y_train = train[target].values

X_validation = validation[features].values
y_validation = validation[target].values

X_test = test[features].values
y_test = test[target].values

## Feature Scaling

Neural networks generally train more effectively when input features are on similar numerical scales.

CCT is measured in micrometres (~480–580), while iCare IOP is measured in mmHg (~3–70).

We therefore standardize both features using:

z = (x − mean) / standard deviation

The scaler is fitted only on the training data. The same transformation is then applied to validation and test data to prevent data leakage.

In [19]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

## Convert Data to PyTorch Tensors

PyTorch neural networks operate on tensors rather than NumPy arrays.

We convert the scaled feature data and target values into PyTorch tensors.

In [20]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)

X_validation_tensor = torch.tensor(
    X_validation_scaled,
    dtype=torch.float32
)
y_validation_tensor = torch.tensor(
    y_validation,
    dtype=torch.float32
).reshape(-1, 1)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)
y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
).reshape(-1, 1)

## Define the Neural Network

The first ANN uses a simple fully connected architecture:

2 inputs → 32 neurons → 16 neurons → 1 output

The two input neurons represent:

- iCare IOP
- CCT

The final output is the predicted Applanation/GAT IOP.

ReLU activation functions are used between the hidden layers to allow the network to learn nonlinear relationships.

In [22]:
class IOP_ANN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = IOP_ANN().to(device)

print(model)
print("Device:", device)

## Loss Function and Optimizer

The neural network needs a way to measure how far its predictions are from the actual GAT IOP values.

We use Mean Squared Error (MSE) as the training loss.

The optimizer then adjusts the network's weights to reduce this loss.

We use the Adam optimizer for the initial model.

In [24]:
loss_function = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

## Prepare Training Batches

The training data is divided into smaller batches.

Instead of sending all 4,648 samples through the network at once, the ANN processes a fixed number of samples at a time.

After each batch, the model calculates its error and updates its weights.

In [25]:
batch_size = 32

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

In [26]:
for X_batch, y_batch in train_loader:

    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)

    optimizer.zero_grad()

    predictions = model(X_batch)

    loss = loss_function(predictions, y_batch)

    loss.backward()

    optimizer.step()

## Train the Neural Network

An epoch is one complete pass through the entire training dataset.

During each epoch, the model:

1. Processes the training data in batches.
2. Makes predictions.
3. Calculates the MSE loss.
4. Uses backpropagation to calculate gradients.
5. Uses Adam to update the weights.

We record the average training loss after each epoch so that we can monitor the learning process.

In [ ]:
epochs = 200

train_losses = []

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(X_batch)

        # Calculate loss
        loss = loss_function(predictions, y_batch)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    train_losses.append(average_loss)

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"- Loss: {average_loss:.4f}"
        )

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_losses)

plt.xlabel("Epoch")
plt.ylabel("Training MSE Loss")
plt.title("ANN Training Loss")

plt.grid(True, alpha=0.3)
plt.show()

## Initial ANN Validation Evaluation

We now evaluate the trained ANN on the validation dataset using the same metrics used for the Random Forest.

The test dataset remains untouched.

In [ ]:
model.eval()

with torch.no_grad():
    validation_predictions_tensor = model(
        X_validation_tensor.to(device)
    )

validation_predictions_ann = (
    validation_predictions_tensor
    .cpu()
    .numpy()
    .flatten()
)

ann_val_mae = mean_absolute_error(
    y_validation,
    validation_predictions_ann
)

ann_val_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        validation_predictions_ann
    )
)

ann_val_r2 = r2_score(
    y_validation,
    validation_predictions_ann
)

print("ANN Validation Results")
print("------------------")
print(f"MAE:  {ann_val_mae:.3f} mmHg")
print(f"RMSE: {ann_val_rmse:.3f} mmHg")
print(f"R²:   {ann_val_r2:.3f}")

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_losses)

plt.xlabel("Epoch")
plt.ylabel("Training MSE Loss")
plt.title("ANN Training Loss")

plt.grid()
plt.show()

## Learning Rate Experiment

The initial ANN used a learning rate of 0.001.

The training loss reached a plateau, so we will test different learning rates while keeping the ANN architecture and dataset unchanged.

The test dataset remains untouched.

CONCLUSION: 
The learning-rate experiment showed that a learning rate of 0.001 performs best.

In [ ]:
learning_rates = [0.01, 0.001, 0.0001, 0.00001]

lr_results = []

for lr in learning_rates:

    # Create a fresh model for each experiment
    model = IOP_ANN().to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    epochs = 200

    for epoch in range(epochs):

        model.train()

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            predictions = model(X_batch)

            loss = loss_function(
                predictions,
                y_batch
            )

            loss.backward()
            optimizer.step()

    # Validation prediction
    model.eval()

    with torch.no_grad():
        validation_predictions = model(
            X_validation_tensor.to(device)
        )

    validation_predictions = (
        validation_predictions
        .cpu()
        .numpy()
        .flatten()
    )

    mae = mean_absolute_error(
        y_validation,
        validation_predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_validation,
            validation_predictions
        )
    )

    r2 = r2_score(
        y_validation,
        validation_predictions
    )

    lr_results.append(
        [lr, mae, rmse, r2]
    )

    print(f"Learning rate: {lr}")
    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²:   {r2:.3f}")
    print()

## ANN Architecture Experiment

The learning-rate experiment showed that a learning rate of 0.001 performs best.

We now test different network architectures while keeping the learning rate, optimizer, loss function, batch size, epochs, and datasets unchanged.

The goal is to determine whether increasing or decreasing network capacity improves validation performance.

The test dataset remains untouched.

In [ ]:
architectures = {
    "16": [16],
    "32-16": [32, 16],
    "64-32": [64, 32],
    "64-32-16": [64, 32, 16]
}

architecture_results = []

for name, hidden_layers in architectures.items():

    layers = []
    input_size = 2

    # Build hidden layers
    for hidden_size in hidden_layers:
        layers.append(nn.Linear(input_size, hidden_size))
        layers.append(nn.ReLU())
        input_size = hidden_size

    # Output layer
    layers.append(nn.Linear(input_size, 1))

    network = nn.Sequential(*layers).to(device)

    optimizer = torch.optim.Adam(
        network.parameters(),
        lr=0.001
    )

    # Train
    for epoch in range(200):

        network.train()

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            predictions = network(X_batch)

            loss = loss_function(
                predictions,
                y_batch
            )

            loss.backward()
            optimizer.step()

    # Validation
    network.eval()

    with torch.no_grad():
        predictions = network(
            X_validation_tensor.to(device)
        )

    predictions = (
        predictions
        .cpu()
        .numpy()
        .flatten()
    )

    mae = mean_absolute_error(
        y_validation,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_validation,
            predictions
        )
    )

    r2 = r2_score(
        y_validation,
        predictions
    )

    architecture_results.append(
        [name, mae, rmse, r2]
    )

    print(f"Architecture: 2 → {name} → 1")
    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²:   {r2:.3f}")
    print()

In [ ]:
architecture_results_df = pd.DataFrame(
    architecture_results,
    columns=["Architecture", "MAE", "RMSE", "R²"]
)

architecture_results_df

## ANN Activation Function Experiment

The architecture experiment showed that increasing network size does not produce a meaningful improvement.

We now test different activation functions while keeping the architecture, learning rate, optimizer, loss function, batch size, epochs, and datasets unchanged.

The test dataset remains untouched.

In [ ]:
activations = {
    "ReLU": nn.ReLU(),
    "Tanh": nn.Tanh(),
    "LeakyReLU": nn.LeakyReLU()
}

activation_results = []

for name, activation in activations.items():

    network = nn.Sequential(
        nn.Linear(2, 32),
        activation,
        nn.Linear(32, 16),
        activation,
        nn.Linear(16, 1)
    ).to(device)

    optimizer = torch.optim.Adam(
        network.parameters(),
        lr=0.001
    )

    # Train
    for epoch in range(200):

        network.train()

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            predictions = network(X_batch)

            loss = loss_function(
                predictions,
                y_batch
            )

            loss.backward()
            optimizer.step()

    # Validation
    network.eval()

    with torch.no_grad():
        predictions = network(
            X_validation_tensor.to(device)
        )

    predictions = (
        predictions
        .cpu()
        .numpy()
        .flatten()
    )

    mae = mean_absolute_error(
        y_validation,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_validation,
            predictions
        )
    )

    r2 = r2_score(
        y_validation,
        predictions
    )

    activation_results.append(
        [name, mae, rmse, r2]
    )

    print(f"Activation: {name}")
    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²:   {r2:.3f}")
    print()

In [36]:
activation_results_df = pd.DataFrame(
    activation_results,
    columns=["Activation", "MAE", "RMSE", "R²"]
)

activation_results_df

,Activation,MAE,RMSE,R²
0,ReLU,2.833026,3.993462,0.677307
1,Tanh,2.808187,4.001266,0.676045
2,LeakyReLU,2.808879,3.982930,0.679007


## ANN Error Analysis by IOP Range

Overall validation metrics can hide differences in performance across the IOP range.

We therefore examine MAE and RMSE separately across clinically relevant GAT IOP ranges.

The test dataset remains untouched.

In [37]:
best_ann = nn.Sequential(
    nn.Linear(2, 32),
    nn.LeakyReLU(),
    nn.Linear(32, 16),
    nn.LeakyReLU(),
    nn.Linear(16, 1)
).to(device)

In [38]:
optimizer = torch.optim.Adam(
    best_ann.parameters(),
    lr=0.001
)

for epoch in range(200):

    best_ann.train()

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        predictions = best_ann(X_batch)

        loss = loss_function(
            predictions,
            y_batch
        )

        loss.backward()
        optimizer.step()

In [39]:
best_ann.eval()

with torch.no_grad():
    ann_predictions = best_ann(
        X_validation_tensor.to(device)
    )

ann_predictions = (
    ann_predictions
    .cpu()
    .numpy()
    .flatten()
)

In [40]:
validation_analysis = validation.copy()

validation_analysis["prediction"] = ann_predictions

validation_analysis["error"] = (
    validation_analysis["applanation_iop"]
    - validation_analysis["prediction"]
)

validation_analysis["abs_error"] = (
    validation_analysis["error"].abs()
)

validation_analysis["IOP_range"] = pd.cut(
    validation_analysis["applanation_iop"],
    bins=[0, 10, 20, 30, 40, np.inf],
    labels=["<10", "10-20", "20-30", "30-40", "40+"]
)

range_results = (
    validation_analysis
    .groupby("IOP_range", observed=False)
    .agg(
        Samples=("abs_error", "count"),
        MAE=("abs_error", "mean"),
        RMSE=("error", lambda x: np.sqrt(np.mean(x**2)))
    )
)

range_results

,Samples,MAE,RMSE
IOP_range,,,
<10,45,3.474608,3.941435
10-20,703,2.088384,2.797133
20-30,192,3.709857,4.675011
30-40,46,7.455769,8.674369
40+,15,9.627094,11.857982


## Activation Function Experiment — Conclusion

Changing the activation function produced no meaningful improvement in validation performance.

ReLU, Tanh, and LeakyReLU all produced validation MAE values of approximately 2.8 mmHg. Tanh achieved the lowest MAE at 2.808 mmHg, while LeakyReLU achieved the lowest RMSE and highest R², but these differences are very small.

Therefore, activation function does not appear to be a major limiting factor for this ANN.

The ANN remains at approximately 2.8 mmHg validation MAE, similar to the Random Forest model.

## Final Controlled ANN Experiment

Previous experiments showed that learning rate, network architecture, and activation function produced only small differences in validation performance.

We now perform a controlled comparison of the strongest architectures and activation functions using a fixed random seed.

All other training settings remain unchanged:

- Learning rate: 0.001
- Optimizer: Adam
- Loss: MSE
- Batch size: 32
- Epochs: 200
- Features: iCare IOP and CCT
- Target: Applanation/GAT IOP

The test dataset remains untouched.

In [41]:
architectures = {
    "16": [16],
    "32-16": [32, 16],
    "64-32": [64, 32]
}

activation_functions = {
    "ReLU": nn.ReLU,
    "Tanh": nn.Tanh,
    "LeakyReLU": nn.LeakyReLU
}

controlled_results = []

for architecture_name, hidden_layers in architectures.items():

    for activation_name, activation_function in activation_functions.items():

        # Reset random seed for every experiment
        torch.manual_seed(42)
        np.random.seed(42)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(42)

        # Build network
        layers = []
        input_size = 2

        for hidden_size in hidden_layers:
            layers.append(
                nn.Linear(input_size, hidden_size)
            )
            layers.append(
                activation_function()
            )
            input_size = hidden_size

        layers.append(
            nn.Linear(input_size, 1)
        )

        network = nn.Sequential(*layers).to(device)

        optimizer = torch.optim.Adam(
            network.parameters(),
            lr=0.001
        )

        # Train
        for epoch in range(200):

            network.train()

            for X_batch, y_batch in train_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                optimizer.zero_grad()

                predictions = network(X_batch)

                loss = loss_function(
                    predictions,
                    y_batch
                )

                loss.backward()
                optimizer.step()

        # Validation
        network.eval()

        with torch.no_grad():
            predictions = network(
                X_validation_tensor.to(device)
            )

        predictions = (
            predictions
            .cpu()
            .numpy()
            .flatten()
        )

        mae = mean_absolute_error(
            y_validation,
            predictions
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                predictions
            )
        )

        r2 = r2_score(
            y_validation,
            predictions
        )

        controlled_results.append([
            architecture_name,
            activation_name,
            mae,
            rmse,
            r2
        ])

controlled_results_df = pd.DataFrame(
    controlled_results,
    columns=[
        "Architecture",
        "Activation",
        "MAE",
        "RMSE",
        "R²"
    ]
)

controlled_results_df.sort_values("MAE")

,Architecture,Activation,MAE,RMSE,R²
4,32-16,Tanh,2.806794,4.012825,0.674170
5,32-16,LeakyReLU,2.814394,3.988437,0.678119
3,32-16,ReLU,2.825363,3.988653,0.678084
6,64-32,ReLU,2.827025,3.994466,0.677145
8,64-32,LeakyReLU,2.829177,3.993144,0.677359
1,16,Tanh,2.831011,4.002750,0.675805
7,64-32,Tanh,2.835595,4.004054,0.675593
0,16,ReLU,2.838221,4.005821,0.675307
2,16,LeakyReLU,2.843017,4.011773,0.674341


Across different learning rates, network sizes, and activation functions, the ANN consistently produces approximately 2.8 mmHg validation MAE. The best configuration does not meaningfully outperform the Random Forest, which achieved approximately 2.81 mmHg validation MAE.